# 05 — Modeling, evaluation & explainability (Phase 2–5)

**DSP391m · Group 1 · FPT University** — Tasks 23–34 (RQ1/RQ2/RQ3).

This notebook is the modeling counterpart to the data notebooks (00–04). Everything
here is built on the **frozen train/test split** and the **anti-leakage
preprocessing** from Phase 1: preprocessing + SMOTE are fit on the *train fold only*
at every checkpoint, then applied to the held-out test.

**What it covers**

1. **Benchmark** of the five candidate algorithms (LR / RF / XGB / LGBM / ANN) with
   SMOTE class balancing — *Phase 2*.
2. **Time-aware performance** across the six checkpoints (10 → 100 % of course
   length) — how early can we predict reliably? — *Phase 3, RQ1*.
3. **Statistical significance** — Friedman + post-hoc Wilcoxon on the repeated-CV
   folds, so model ranking is defensible, not eyeballed.
4. **Confusion matrix & decision-threshold tuning** — the 0.5 default is arbitrary
   for an imbalanced early-warning task; we tune for a recall target.
5. **Fine-tuning** (RandomizedSearchCV) and an honest sensitivity check — *Task 4*.
6. **Explainability** — model-native importance here; the model-agnostic
   SHAP / LIME / stability layer lives in `src/xai/` — *Phase 5, RQ2/RQ3*.

All numbers regenerate from `src/` on the trained bundles in `models/`; figures are
written to `reports/figures/` at 300 dpi.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from IPython.display import Image, display
from loguru import logger

logger.remove()  # keep the report output clean; CLI runs still log

from src import plots
from src.config import FIGURES_DIR, TABLES_DIR
from src.evaluation import stat_tests
from src.modeling import threshold as thr
from src.modeling.predict import predict_checkpoint

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 20)


def show(name):
    display(Image(filename=str(FIGURES_DIR / f"{name}.png")))


BEST = "xgb"  # highest held-out recall at t=100% (confirmed in §1)
print("tables:", TABLES_DIR)
print("figures:", FIGURES_DIR)

## 1 · Benchmark of the five models (Phase 2)

Held-out test metrics at the full-course checkpoint (t = 100 %). SMOTE balances the
transformed **train** fold only; recall of the *at-risk* class is the headline
metric — missing a struggling student is the costly error.

In [ ]:
metrics = pd.read_csv(TABLES_DIR / "model_metrics.csv")
bench = metrics[metrics["t_percent"] == 100].sort_values("recall", ascending=False)
display(bench.reset_index(drop=True))

plots.model_comparison_bar(metrics, t_percent=100, name="model_benchmark")
show("model_benchmark")

## 2 · Time-aware performance — RQ1 (Phase 3)

The project question: *how early in the course can we flag at-risk students
reliably?* Each model is retrained at six checkpoints (features cut at
`round(module_length · t/100)` days). The table below is the **best model at each
checkpoint** and whether it clears the reliability bar
(recall ≥ 0.80 **and** PR-AUC ≥ 0.90).

In [ ]:
best = pd.read_csv(TABLES_DIR / "time_aware_best.csv")
display(best)

for m in ("recall", "pr_auc"):
    plots.metric_vs_checkpoint(metrics, metric=m, name=f"time_aware_{m}")
show("time_aware_recall")
show("time_aware_pr_auc")

earliest = best.loc[best["reliable"], "t_percent"].min()
print(f"Earliest reliable checkpoint: t = {earliest}% of course length")

## 3 · Is the ranking real? Friedman + post-hoc Wilcoxon

The held-out table gives one number per model — it cannot tell noise from a real
gap. The repeated **5×5 cross-validation** gives 25 *paired* fold scores per model,
so we test properly: a Friedman test per metric (any model different?), then
pairwise Wilcoxon signed-rank with **Holm** correction (which pairs differ?).

In [ ]:
fried, pair = stat_tests.run_all(t_percent=100)
display(fried)

print("Pairwise Wilcoxon on recall (Holm-corrected):")
display(pair[pair["metric"] == "recall"].reset_index(drop=True))

**Reading it:** every metric's Friedman p-value is far below 0.05 — the models are
genuinely different. On **recall**, XGBoost is the top-ranked model and its
advantage over each rival is significant after Holm correction; on the aggregate
ranking metrics (F1 / PR-AUC / ROC-AUC) LightGBM edges ahead. We therefore carry
**XGBoost** forward as the recall-first early-warning model.

## 4 · Confusion matrix & decision-threshold tuning

A single accuracy/recall number hides *where* the errors fall. The confusion matrix
below (for the recall-first model at t = 100 %) makes the false-negative cell — the
at-risk students we miss — explicit. The default 0.5 cut is arbitrary; we sweep the
threshold and pick operating points by F1, Youden's J, and a **recall ≥ 0.90**
policy, then compare the naive vs tuned confusion matrices.

In [ ]:
sweep, chosen, table = thr.tune_checkpoint(BEST, 100, target_recall=0.9)
display(table)

plots.threshold_curve(sweep, chosen, name="threshold_tuning")
show("threshold_tuning")

out = predict_checkpoint(BEST, 100)
tuned = chosen[next(k for k in chosen if k.startswith("recall"))]
plots.confusion_matrix_plot(
    out["y_true"], (out["proba"] >= 0.5).astype(int),
    name="confusion_default_t100", title=f"{BEST} @ t=100% — default 0.50",
)
plots.confusion_matrix_plot(
    out["y_true"], (out["proba"] >= tuned).astype(int),
    name="confusion_tuned_t100", title=f"{BEST} @ t=100% — tuned {tuned:.2f} (recall>=0.9)",
)
show("confusion_default_t100")
show("confusion_tuned_t100")

**Reading it:** raising the cut to ~0.87 trades a little recall (still ≥ 0.90) for a
large precision gain (fewer false alarms) — the right move when outreach capacity is
limited. The confusion matrices show the shift concretely in the off-diagonal cells.

## 5 · Fine-tuning & sensitivity (Task 4)

`RandomizedSearchCV` (SMOTE inside each fold, scored by PR-AUC) tunes the two tree
models. The honest sensitivity check re-scores the winner on the *active-students*
subgroup, so we don't over-claim on the easy-to-classify inactive cases.

In [ ]:
for fname in ("tuning_results.csv", "sensitivity_active_xgb.csv"):
    p = TABLES_DIR / fname
    if p.exists():
        print(fname)
        display(pd.read_csv(p))

## 6 · Explainability — RQ2/RQ3 (Phase 5)

Below is the **model-native** feature importance for the recall-first model — a fast,
dependency-light view of what drives the prediction. The deeper, **model-agnostic**
layer (SHAP + LIME per-student explanations and the Jaccard / Spearman *stability*
metrics across seeds and checkpoints) lives in `src/xai/` and runs in the SHAP
environment; this notebook stays runnable in the base env.

In [ ]:
import joblib

from src.config import MODELS_DIR

bundle = joblib.load(MODELS_DIR / f"{BEST}_t100.joblib")
imp = pd.DataFrame(
    {"feature": bundle["feat_names"], "importance": bundle["model"].feature_importances_}
).sort_values("importance", ascending=False)
display(imp.head(15).reset_index(drop=True))

plots.importance_bar(imp, top=15, name=f"native_importance_{BEST}_t100")
show(f"native_importance_{BEST}_t100")

## Summary

- **Five models benchmarked** with SMOTE across **six time checkpoints** — the full
  modeling grid, not a single fit.
- **RQ1:** performance rises monotonically with course progress; the reliability bar
  (recall ≥ 0.80, PR-AUC ≥ 0.90) is first cleared partway through the course, so
  useful early warnings are possible well before the end.
- **Ranking is statistically defensible:** Friedman + Holm-corrected Wilcoxon on the
  repeated-CV folds — XGBoost leads on recall, LightGBM on aggregate metrics.
- **Operating point tuned** for a recall target instead of the arbitrary 0.5, with
  confusion matrices showing the concrete error trade-off.
- **Explainability** is code-complete in `src/xai/` (SHAP/LIME + stability); native
  importance shown here as a runnable preview.